In [0]:
# Configuración personalizada por tabla
tablas_a_validar = {
    "db_ventas.fact_ventas": {
        "claves": ["id_venta"],
        "categorias": {"estado_venta": ["COMPRADO", "PENDIENTE", "CANCELADO"]},
    },
    "db_clientes.dim_clientes": {
        "claves": ["id_cliente"],
        "categorias": {"tipo_cliente": ["PLATINUM", "GOLD", "REGULAR"]},
    },
    "db_finanzas.pagos_cuotas": {
        "claves": ["id_pago", "num_cuota"],
        "categorias": {"metodo_pago": ["TARJETA", "EFECTIVO", "TRANSFERENCIA"]},
    },
}

# Ejecución masiva del pipeline de calidad
for tabla, config in tablas_a_validar.items():
    validar_calidad_tabla_delta(
        spark_session=spark,
        tabla_nombre=tabla,
        claves_primarias=config.get("claves"),
        categorias_validas=config.get("categorias"),
    )


In [0]:
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import StringType


def validar_calidad_tabla_delta(
    spark_session,
    tabla_nombre,
    claves_primarias=None,
    columnas_fecha=None,
    reglas_numericas=None,
    categorias_validas=None
):
    """
    Analiza la calidad de una tabla Delta sin modificarla.

    Parámetros
    ----------
    claves_primarias:
        Lista de columnas que forman la posible clave primaria.

    columnas_fecha:
        Diccionario con reglas para columnas de fecha.
        Ejemplo:
        {
            "order_purchase_timestamp": {
                "tipo": "timestamp",
                "min": "2010-01-01",
                "max": "2030-12-31"
            }
        }

    reglas_numericas:
        Diccionario con límites permitidos.
        Ejemplo:
        {
            "price": {"min": 0},
            "review_score": {"min": 1, "max": 5}
        }

    categorias_validas:
        Diccionario con valores permitidos.
    """

    claves_primarias = claves_primarias or []
    columnas_fecha = columnas_fecha or {}
    reglas_numericas = reglas_numericas or {}
    categorias_validas = categorias_validas or {}

    df = spark_session.table(tabla_nombre)
    total_filas = df.count()

    # Validar que las columnas configuradas existan
    columnas_configuradas = set(
        claves_primarias
        + list(columnas_fecha.keys())
        + list(reglas_numericas.keys())
        + list(categorias_validas.keys())
    )

    columnas_inexistentes = columnas_configuradas - set(df.columns)

    if columnas_inexistentes:
        raise ValueError(
            f"Las siguientes columnas no existen en {tabla_nombre}: "
            f"{sorted(columnas_inexistentes)}"
        )

    resultados = {}

    resumen_metricas = [
        ("tabla", tabla_nombre),
        ("total_filas", str(total_filas)),
        ("total_columnas", str(len(df.columns)))
    ]

    if total_filas == 0:
        resultados["resumen"] = spark_session.createDataFrame(
            resumen_metricas,
            "metrica string, valor string"
        )
        return resultados

    # -------------------------------------------------------
    # 1. Nulos y cadenas vacías
    # -------------------------------------------------------

    expresiones_nulos = []

    for campo in df.schema.fields:
        columna = campo.name
        condicion = F.col(columna).isNull()

        if isinstance(campo.dataType, StringType):
            condicion = condicion | (
                F.trim(F.col(columna)) == ""
            )

        expresiones_nulos.append(
            F.sum(
                F.when(condicion, 1).otherwise(0)
            ).alias(columna)
        )

    cantidades_nulos = df.agg(
        *expresiones_nulos
    ).first().asDict()

    filas_nulos = [
        (
            columna,
            int(cantidad),
            round((cantidad / total_filas) * 100, 2)
        )
        for columna, cantidad in cantidades_nulos.items()
    ]

    resultados["nulos_y_vacios"] = spark_session.createDataFrame(
        filas_nulos,
        "columna string, cantidad long, porcentaje double"
    ).orderBy(F.desc("porcentaje"))

    # -------------------------------------------------------
    # 2. Filas completamente duplicadas
    # -------------------------------------------------------

    duplicados_completos = (
        df.groupBy(*df.columns)
        .count()
        .filter(F.col("count") > 1)
    )

    cantidad_duplicados = (
        duplicados_completos
        .select(
            F.sum(F.col("count") - 1).alias("duplicados")
        )
        .first()["duplicados"]
        or 0
    )

    resumen_metricas.append(
        ("filas_duplicadas_completas", str(cantidad_duplicados))
    )

    resultados["duplicados_completos"] = duplicados_completos

    # -------------------------------------------------------
    # 3. Calidad de posibles claves primarias
    # -------------------------------------------------------

    if claves_primarias:
        condicion_clave_nula = reduce(
            lambda acumulado, condicion:
                acumulado | condicion,
            [
                F.col(columna).isNull()
                for columna in claves_primarias
            ]
        )

        filas_clave_nula = df.filter(
            condicion_clave_nula
        ).count()

        claves_duplicadas = (
            df.groupBy(*claves_primarias)
            .count()
            .filter(F.col("count") > 1)
        )

        grupos_clave_duplicados = claves_duplicadas.count()

        resumen_metricas.extend([
            (
                "filas_con_clave_nula",
                str(filas_clave_nula)
            ),
            (
                "grupos_de_claves_duplicadas",
                str(grupos_clave_duplicados)
            )
        ])

        resultados["claves_duplicadas"] = claves_duplicadas

    # -------------------------------------------------------
    # 4. Validación de fechas
    # -------------------------------------------------------

    filas_fechas = []

    for columna, regla in columnas_fecha.items():
        tipo = regla.get("tipo", "timestamp")
        minimo = regla.get("min")
        maximo = regla.get("max")

        if tipo == "date":
            valor_convertido = F.to_date(F.col(columna))
        else:
            valor_convertido = F.to_timestamp(F.col(columna))

        df_fecha = df.select(
            F.col(columna).alias("valor_original"),
            valor_convertido.alias("valor_convertido")
        )

        errores_conversion = df_fecha.filter(
            F.col("valor_original").isNotNull()
            & F.col("valor_convertido").isNull()
        ).count()

        fuera_minimo = 0
        fuera_maximo = 0

        if minimo:
            limite_minimo = (
                F.to_date(F.lit(minimo))
                if tipo == "date"
                else F.to_timestamp(F.lit(minimo))
            )

            fuera_minimo = df_fecha.filter(
                F.col("valor_convertido").isNotNull()
                & (F.col("valor_convertido") < limite_minimo)
            ).count()

        if maximo:
            limite_maximo = (
                F.to_date(F.lit(maximo))
                if tipo == "date"
                else F.to_timestamp(F.lit(maximo))
            )

            fuera_maximo = df_fecha.filter(
                F.col("valor_convertido").isNotNull()
                & (F.col("valor_convertido") > limite_maximo)
            ).count()

        filas_fechas.append(
            (
                columna,
                tipo,
                errores_conversion,
                fuera_minimo,
                fuera_maximo
            )
        )

    resultados["validacion_fechas"] = spark_session.createDataFrame(
        filas_fechas,
        """
        columna string,
        tipo_esperado string,
        errores_conversion long,
        anteriores_al_minimo long,
        posteriores_al_maximo long
        """
    )

    # -------------------------------------------------------
    # 5. Validación numérica
    # -------------------------------------------------------

    filas_numericas = []

    for columna, regla in reglas_numericas.items():
        minimo = regla.get("min")
        maximo = regla.get("max")

        valor_numerico = F.expr(
            f"try_cast(`{columna}` AS DOUBLE)"
        )

        df_numero = df.select(
            F.col(columna).alias("valor_original"),
            valor_numerico.alias("valor_numerico")
        )

        errores_conversion = df_numero.filter(
            F.col("valor_original").isNotNull()
            & F.col("valor_numerico").isNull()
        ).count()

        inferiores_minimo = 0
        superiores_maximo = 0

        if minimo is not None:
            inferiores_minimo = df_numero.filter(
                F.col("valor_numerico") < minimo
            ).count()

        if maximo is not None:
            superiores_maximo = df_numero.filter(
                F.col("valor_numerico") > maximo
            ).count()

        filas_numericas.append(
            (
                columna,
                errores_conversion,
                inferiores_minimo,
                superiores_maximo
            )
        )

    resultados["validacion_numerica"] = spark_session.createDataFrame(
        filas_numericas,
        """
        columna string,
        errores_conversion long,
        inferiores_al_minimo long,
        superiores_al_maximo long
        """
    )

    # -------------------------------------------------------
    # 6. Categorías inesperadas
    # -------------------------------------------------------

    resumen_categorias = []
    detalles_categorias = {}

    for columna, valores_permitidos in categorias_validas.items():
        valores_inesperados = (
            df.filter(
                F.col(columna).isNotNull()
                & ~F.col(columna).isin(valores_permitidos)
            )
            .groupBy(columna)
            .count()
            .orderBy(F.desc("count"))
        )

        cantidad_inesperados = (
            valores_inesperados
            .select(F.sum("count").alias("total"))
            .first()["total"]
            or 0
        )

        resumen_categorias.append(
            (columna, cantidad_inesperados)
        )

        detalles_categorias[columna] = valores_inesperados

    resultados["resumen_categorias"] = (
        spark_session.createDataFrame(
            resumen_categorias,
            "columna string, cantidad_inesperada long"
        )
    )

    resultados["detalle_categorias"] = detalles_categorias

    # Resumen general
    resultados["resumen"] = spark_session.createDataFrame(
        resumen_metricas,
        "metrica string, valor string"
    )

    return resultados

In [0]:
def mostrar_resultados_calidad(resultados):
    for nombre, resultado in resultados.items():
        print(f"\n{'=' * 70}")
        print(nombre.upper())
        print("=" * 70)

        if isinstance(resultado, dict):
            for columna, detalle in resultado.items():
                print(f"\nValores inesperados en: {columna}")
                display(detalle)
        else:
            display(resultado)

In [0]:
configuracion_calidad = {
    "olist.bronze.customers": {
        "claves_primarias": ["customer_id"],
        "reglas_numericas": {
            "customer_zip_code_prefix": {
                "min": 0,
                "max": 99999
            }
        }
    },

    "olist.bronze.geolocation": {
        "reglas_numericas": {
            "geolocation_zip_code_prefix": {
                "min": 0,
                "max": 99999
            },
            "geolocation_lat": {
                "min": -90,
                "max": 90
            },
            "geolocation_lng": {
                "min": -180,
                "max": 180
            }
        }
    },

    "olist.bronze.order_items": {
        "claves_primarias": [
            "order_id",
            "order_item_id"
        ],
        "columnas_fecha": {
            "shipping_limit_date": {
                "tipo": "timestamp",
                "min": "2010-01-01",
                "max": "2030-12-31"
            }
        },
        "reglas_numericas": {
            "order_item_id": {"min": 1},
            "price": {"min": 0},
            "freight_value": {"min": 0}
        }
    },

    "olist.bronze.order_payments": {
        "claves_primarias": [
            "order_id",
            "payment_sequential"
        ],
        "reglas_numericas": {
            "payment_sequential": {"min": 1},
            "payment_installments": {"min": 1},
            "payment_value": {"min": 0}
        },
        "categorias_validas": {
            "payment_type": [
                "credit_card",
                "boleto",
                "voucher",
                "debit_card",
                "not_defined"
            ]
        }
    },

    "olist.bronze.order_reviews": {
        "claves_primarias": [
            "review_id",
            "order_id"
        ],
        "columnas_fecha": {
            "review_creation_date": {
                "tipo": "timestamp"
            },
            "review_answer_timestamp": {
                "tipo": "timestamp"
            }
        },
        "reglas_numericas": {
            "review_score": {
                "min": 1,
                "max": 5
            }
        }
    },

    "olist.bronze.orders": {
        "claves_primarias": ["order_id"],
        "columnas_fecha": {
            "order_purchase_timestamp": {
                "tipo": "timestamp"
            },
            "order_approved_at": {
                "tipo": "timestamp"
            },
            "order_delivered_carrier_date": {
                "tipo": "timestamp"
            },
            "order_delivered_customer_date": {
                "tipo": "timestamp"
            },
            "order_estimated_delivery_date": {
                "tipo": "timestamp"
            }
        },
        "categorias_validas": {
            "order_status": [
                "created",
                "approved",
                "invoiced",
                "processing",
                "shipped",
                "delivered",
                "unavailable",
                "canceled"
            ]
        }
    },

    "olist.bronze.products": {
        "claves_primarias": ["product_id"],
        "reglas_numericas": {
            "product_name_lenght": {"min": 0},
            "product_description_lenght": {"min": 0},
            "product_photos_qty": {"min": 0},
            "product_weight_g": {"min": 0},
            "product_length_cm": {"min": 0},
            "product_height_cm": {"min": 0},
            "product_width_cm": {"min": 0}
        }
    },

    "olist.bronze.sellers": {
        "claves_primarias": ["seller_id"],
        "reglas_numericas": {
            "seller_zip_code_prefix": {
                "min": 0,
                "max": 99999
            }
        }
    },

    "olist.bronze.product_category_translation": {
        "claves_primarias": [
            "product_category_name"
        ]
    }
}

In [0]:
tabla_customers = "olist.bronze.customers"

resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_customers,
    **configuracion_calidad[tabla_customers]
)

mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_geolocation = "olist.bronze.geolocation"

resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_geolocation,
    **configuracion_calidad[tabla_geolocation]
)

mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_order_items = "olist.bronze.order_items"

resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_order_items,
    **configuracion_calidad[tabla_order_items]
)

mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_order_payments = "olist.bronze.order_payments"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_order_payments,
    **configuracion_calidad[tabla_order_payments]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_order_reviers = "olist.bronze.order_reviews"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_order_reviers,
    **configuracion_calidad[tabla_order_reviers]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_orders = "olist.bronze.orders"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_orders,
    **configuracion_calidad[tabla_orders]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_product_category_translation = "olist.bronze.product_category_translation"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_product_category_translation,
    **configuracion_calidad[tabla_product_category_translation]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_products = "olist.bronze.products"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_products,
    **configuracion_calidad[tabla_products]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
tabla_sellers = "olist.bronze.sellers"
resultado_customers = validar_calidad_tabla_delta(
    spark_session=spark,
    tabla_nombre=tabla_sellers,
    **configuracion_calidad[tabla_sellers]
)
mostrar_resultados_calidad(resultado_customers)

In [0]:
def validar_integridad_referencial(
    spark_session,
    tabla_hija,
    columnas_hija,
    tabla_padre,
    columnas_padre
):
    if len(columnas_hija) != len(columnas_padre):
        raise ValueError(
            "Las claves de ambas tablas deben tener "
            "la misma cantidad de columnas."
        )

    hija = spark_session.table(tabla_hija).alias("hija")

    padre = (
        spark_session.table(tabla_padre)
        .select(
            *[
                F.col(columna).alias(f"clave_padre_{indice}")
                for indice, columna in enumerate(columnas_padre)
            ]
        )
        .dropDuplicates()
        .alias("padre")
    )

    claves_no_nulas = reduce(
        lambda acumulado, condicion:
            acumulado & condicion,
        [
            F.col(f"hija.{columna}").isNotNull()
            for columna in columnas_hija
        ]
    )

    hija = hija.filter(claves_no_nulas)

    condiciones_join = reduce(
        lambda acumulado, condicion:
            acumulado & condicion,
        [
            F.col(f"hija.{columna_hija}")
            == F.col(f"padre.clave_padre_{indice}")
            for indice, columna_hija
            in enumerate(columnas_hija)
        ]
    )

    return hija.join(
        padre,
        condiciones_join,
        "left_anti"
    )

In [0]:
pedidos_sin_cliente = validar_integridad_referencial(
    spark,
    "olist.bronze.orders",
    ["customer_id"],
    "olist.bronze.customers",
    ["customer_id"]
)

display(pedidos_sin_cliente)

In [0]:
# 1. Order items sin pedido existente
order_items_sin_order = validar_integridad_referencial(
    spark,
    "olist.bronze.order_items",
    ["order_id"],
    "olist.bronze.orders",
    ["order_id"]
)

display(order_items_sin_order)

In [0]:
# 2. Order items con producto inexistente
order_items_sin_product = validar_integridad_referencial(
    spark,
    "olist.bronze.order_items",
    ["product_id"],
    "olist.bronze.products",
    ["product_id"]
)

display(order_items_sin_product)

In [0]:
# 3. Order items con vendedor inexistente
order_items_sin_seller = validar_integridad_referencial(
    spark,
    "olist.bronze.order_items",
    ["seller_id"],
    "olist.bronze.sellers",
    ["seller_id"]
)

display(order_items_sin_seller)

In [0]:
# 4. Pagos sin pedido existente
payments_sin_order = validar_integridad_referencial(
    spark,
    "olist.bronze.order_payments",
    ["order_id"],
    "olist.bronze.orders",
    ["order_id"]
)

display(payments_sin_order)

In [0]:
# 5. Reviews sin pedido existente
reviews_sin_order = validar_integridad_referencial(
    spark,
    "olist.bronze.order_reviews",
    ["order_id"],
    "olist.bronze.orders",
    ["order_id"]
)

display(reviews_sin_order)

In [0]:
def validar_orden_temporal(
    spark_session,
    tabla_nombre,
    columna_anterior,
    columna_posterior
):
    df = spark_session.table(tabla_nombre)

    fecha_anterior = F.to_timestamp(
        F.col(columna_anterior)
    )

    fecha_posterior = F.to_timestamp(
        F.col(columna_posterior)
    )

    return df.filter(
        F.col(columna_anterior).isNotNull()
        & F.col(columna_posterior).isNotNull()
        & (fecha_posterior < fecha_anterior)
    )

In [0]:
aprobacion_antes_compra = validar_orden_temporal(
    spark,
    "olist.bronze.orders",
    "order_purchase_timestamp",
    "order_approved_at"
)

display(aprobacion_antes_compra)

In [0]:
envio_antes_aprobacion = validar_orden_temporal(
    spark,
    "olist.bronze.orders",
    "order_approved_at",
    "order_delivered_carrier_date"
)

display(envio_antes_aprobacion)

In [0]:
entrega_antes_envio = validar_orden_temporal(
    spark,
    "olist.bronze.orders",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
)

display(entrega_antes_envio)

In [0]:
entrega_antes_compra = validar_orden_temporal(
    spark,
    "olist.bronze.orders",
    "order_purchase_timestamp",
    "order_delivered_customer_date"
)

display(entrega_antes_compra)

# Resumen de calidad de datos

## Problemas detectados por tabla

### Geolocation
- Se identificaron 261.831 filas completamente duplicadas.
- Decisión para Silver: eliminar únicamente duplicados exactos. No asumir que un código postal debe tener una sola coordenada.

## Integridad referencial

- Se encontraron 2.702 reseñas cuyo `order_id` no existe en la tabla `orders`.
- Decisión para Silver: investigar si son registros huérfanos reales o consecuencia de una lectura incorrecta del CSV.

## Coherencia temporal

- 1.359 registros presentan envío al transportista anterior a la aprobación.
- 23 registros presentan entrega al cliente anterior al envío al transportista.
- Decisión para Silver: conservarlos inicialmente con una marca de inconsistencia para analizarlos, en lugar de eliminarlos automáticamente.

### Order Reviews
- La primera ingesta interpretó incorrectamente comentarios multilínea.
- Se corrigió la lectura del CSV y la tabla Bronze quedó con 99.224 registros.
- Después de la corrección no se detectaron problemas de calidad en la tabla.